In [ ]:
import os
from dotenv import load_dotenv


load_dotenv(".env", override=True)

LM_CONFIG = {
    "model": os.getenv("DSPY_MODEL"),
    "api_key": os.getenv("DSPY_API_KEY"),
    "api_base": os.getenv("DSPY_ENDPOINT"),
    "max_tokens": int(os.getenv("DSPY_MAX_TOKENS", 50_000)),
    "temperature": float(os.getenv("DSPY_TEMPERATURE", 1.0)),
    "cache": True,
}

In [14]:
import dspy
class ProduceGist(dspy.Signature):
    """
    Produce 3 - 5 sentence gist of what this chunk is about, so we can assign it to a heading.
    """
    toc_path: list[str] = dspy.InputField(desc="path down which this chunk has traveled so far in the Table of Contents")
    chunk: str = dspy.InputField()
    gist: str = dspy.OutputField()

class ProduceHeaders(dspy.Signature):
    """
    Produce a list of headers (top-level Table of Contents) for structuring a report on *all* chunk contents.
    Make sure every chunk would belong to exactly one section.
    Ultimately your task is to answer the question: "What are the top reasons of rejections and corresponding recommendations by FDA?"
    """
    toc_path: list[str] = dspy.InputField()
    chunk_summaries: str = dspy.InputField()
    headers: list[str] = dspy.OutputField()

class WriteSection(dspy.Signature):
    """
    Craft a Markdown section, given a path down the table of contents, which ends with this section's specific heading.
    Start the content right beneath that heading: use sub-headings of depth at least +1 relative to the ToC path.
    Your section's content is to be entirely derived from the given list of chunks. That content must be complete but
    very concise, with all necessary knowledge from the chunks reproduced and repetitions or irrelevant details omitted. If a certain heading does not have any content, just omit it.
    """
    toc_path: list[str] = dspy.InputField()
    content_chunks = dspy.InputField()
    section_content = dspy.OutputField()

In [15]:
def massively_summarize(
    toc_path: list[str],
    chunks: list[str],
):
    if len(chunks) < 5 or len(toc_path) >= 3:
        content = dspy.ChainOfThought(WriteSection)(toc_path=toc_path, content_chunks=chunks).section_content
        return f"{toc_path[-1]}\n\n{content}"

    # Produce gist for each chunk
    gister = dspy.ChainOfThought(ProduceGist)
    chunk_summaries = gister.batch([
        dspy.Example(toc_path=toc_path, chunk=chunk).with_inputs("toc_path", "chunk")
        for chunk in chunks
    ])
    chunk_summaries = [summary.gist for summary in chunk_summaries]

    # Get headers for all summary chunks
    produce_headers = dspy.ChainOfThought(ProduceHeaders)
    headers = produce_headers(toc_path=toc_path, chunk_summaries=chunk_summaries).headers

    # Classify each chunk under a header
    classifier = dspy.ChainOfThought(f"toc_path: list[str], chunk -> topic: Literal{headers}")
    topics = classifier.batch([
        dspy.Example(toc_path=toc_path, chunk=chunk).with_inputs("toc_path", "chunk")
        for chunk in chunks
    ])

    sections = {topic: [] for topic in headers}
    for topic, chunk in zip(topics, chunks):
        sections[topic.topic].append(chunk)

    # Recursively summarize each section
    parallel_massively_summarize = massively_summarize
    summarized_sections = [
        parallel_massively_summarize(
            toc_path=toc_path + [topic], 
            chunks=section_chunks
        )
        for topic, section_chunks in sections.items()
    ]

    return toc_path[-1] + "\n\n" + "\n\n".join(summarized_sections)

In [ ]:
import glob
import os

pdfs = glob.glob("data/fda_action_letters/**/*.pdf", recursive=True)
print(len(pdfs))

5


In [ ]:
import pymupdf4llm
chunks = [pymupdf4llm.to_markdown(x) for x in pdfs]

In [35]:
chunks_expanded = []
for x in chunks:
    chunks_expanded.extend(x.split("##"))

In [36]:
len(chunks_expanded)

29

In [ ]:
with dspy.context(lm=dspy.LM(**LM_CONFIG)):
    summmary_md = massively_summarize(
        toc_path=["# Key Themes From FDA Letters"],
        chunks=chunks_expanded,
)

Processed 6 / 6 examples: 100%|██████████| 6/6 [00:02<00:00,  2.98it/s]


In [29]:
from IPython.display import Markdown, display
display(Markdown(summmary_md))

# Key Themes From FDA Letters

Overview of Key Themes in FDA Rejection Letters

Overview of Key Themes in FDA Rejection Letters

<!-- No content available for this section. -->

Issuing Authority: Center for Drug Evaluation and Research (CDER)

<!-- No content available for this section. -->

Application Identification and Documentation

### Application Number Documentation

FDA letters identify the referenced application by including an "APPLICATION NUMBER" heading, followed by the specific number (e.g., 211321Orig1s000).

Insufficient Evidence of Efficacy and Clinical Trial Issues

### Clinical Trial Deficiencies

#### Lack of Clinically Meaningful Efficacy
- The symptomatic coprimary endpoint in trial SHP621-301, while statistically significant, did not represent a clinically meaningful change. The ≥30% DSQ responder threshold lacked evidence for clinical relevance, and anchor-based analyses did not support its meaningfulness.

#### Endpoint Selection and Analysis Issues
- The histologic coprimary endpoint showed a statistically significant response in SHP621-301, but its benefit was unclear due to lack of sustained histologic response in SHP621-302. Short-term histologic improvements did not predict meaningful clinical outcomes.
- The key secondary efficacy endpoint (absolute change in 14-day DSQ combined score) showed only a small treatment difference, and results were not supported by the 7-day PGIS. No treatment difference was observed in the 7-day DSQ score.

#### Insufficient Long-Term Data
- EoE is a chronic condition, requiring demonstration of durable benefit. The phase 3 trial SHP621-302 did not establish long-term efficacy, as it failed to show statistically significant results for relapse prevention on both histology and symptoms.

#### Unreliable Subgroup Analyses
- Post hoc analyses of subjects without prior dilation were inadequate. Dilation status was often self-reported and not confirmed by medical records, making subgroup findings unreliable. Minimal separation between treatment groups was observed in confirmed cases.

### FDA Recommendations for Future Studies

- Design a new, adequate, and well-controlled study with prespecified subpopulations (e.g., dilation-naïve, severe baseline dysphagia).
- Include a treatment period of at least 24 weeks, with a 52-week total duration for safety and durability assessment.
- Use prespecified symptomatic and histologic coprimary endpoints, and propose clinically meaningful within-patient score changes using anchor-based methods.
- Complete the new study before resubmitting the application.

Quality System and Manufacturing Deficiencies

### Management Responsibility (21 CFR 820.20)
The FDA found insufficient detail regarding management control responsibilities across facilities involved in manufacturing the combination product. The applicant must describe specific management roles at Proximagen, UCB, and other relevant sites, outlining how compliance with CGMP requirements (21 CFR Part 4) is assured and specifying each facility's functions.

### Design Controls (21 CFR 820.30)
The applicant did not adequately explain UCB's involvement in design control processes. The FDA requests detailed procedures from UCB demonstrating implementation of design controls, including planning, input/output, review, verification, validation, transfer, changes, and maintenance of the design history file.

### Purchasing Controls (21 CFR 820.50)
Procedures for purchasing controls at UCB were not sufficiently described. The FDA requires a summary delineating purchasing responsibilities at all involved facilities, including supplier evaluation, record maintenance, purchasing data approval, and balancing assessment with receiving acceptance. The applicant must ensure contractor/supplier changes do not affect the final product and describe how purchasing controls are applied.

### Corrective and Preventive Actions (CAPA, 21 CFR 820.100)
The CAPA system's integration across facilities was unclear. The FDA requests a summary of CAPA procedures, specifying each facility's responsibilities. The system should cover identification and analysis of quality data, investigation of nonconformities, corrective/preventive actions, and verification/validation of these actions.

### Production Flow Documentation
The FDA asks for a production flow diagram identifying all manufacturing steps for the combination product and specifying which facility is responsible for each step.

### Acceptance Activities
Clarification is needed on how UCB will integrate into acceptance activities for receiving, in-process, and final product release. The applicant must specify which firm performs each acceptance activity and provide criteria for acceptance/rejection at each stage.

### Facility Inspections
Recent FDA inspections at the UCB manufacturing facility identified deficiencies. Satisfactory resolution of these issues is required before approval.

Labeling, Documentation, and Proprietary Name Requirements

<!-- No content available for this section. -->

Recommendations for Addressing Deficiencies

<!-- No content available for this section. -->

Electronic Signature and Authentication

## Manifestation of Electronic Signatures

FDA letters include a standardized statement indicating that the record was signed electronically. This is followed by the signatory's name, the date and time of signing, and a reference ID. For example:

- JESSICA J LEE, signed on 12/17/2021 at 03:14:12 PM (Reference ID: 4906609)
- ERIC P BASTINGS, signed on 03/29/2019 at 05:57:13 PM (Reference ID: 44 1212136994)

These elements serve to authenticate the electronic record and its signature.